## 1. Setup

In [1]:
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/tokenizerv3.pkl .
# !mkdir ./base-models
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/gqa-rope.pt ./base-models/
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/mla-sin.pt ./base-models/
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/gqa-rope-swiglu-moe.pt ./base-models/
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/swiglu-moe-rope-conv.pt ./base-models/

In [2]:
!pip install -q bert-score rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 702.5 kB/s eta 0:00:00a 0:00:01


In [3]:
import os, gc, copy, time, math, random, re, json, pickle, zipfile, hashlib, itertools
from dataclasses import dataclass, replace, asdict
from collections import Counter

import numpy as np, pandas as pd
from tqdm import tqdm
from rouge_score import rouge_scorer
from bert_score import BERTScorer

import torch, torch.nn as nn, torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader

from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF"))
DEVICE = "cuda"

# ── Data location / field names (check against the EDA cell output) ──
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE6_DIR      = "Phase_6_Summ"
ID_KEY, DOC_KEY, SUMMARY_KEY = "id", "document", "summary"

# True -> tiny subsets + separate output dir, to verify the whole pipeline in minutes
SMOKE_TEST = False
OUT = "./cs781_phase6_smoke" if SMOKE_TEST else "./cs781_phase6"
os.makedirs(OUT, exist_ok=True)

# ── Phase 1 model hyper-parameters (LMConfig reads these) ──
VOCAB_SIZE       = 20_000
TRAIN_BLOCK_SIZE = 2048
N_EMBD           = 768
N_LAYER          = 12
N_HEAD           = 12
N_KV_HEAD        = 4
KV_LATENT_DIM    = N_EMBD // 4

PHASE1_CONFIGS = {
    "gqa-rope":             {"n_layers": 8,  "d_ff": 4 * 768, "attention_type": "gqa", "pos_emb_type": "rope",       "use_moe": False, "use_conv": False},
    "mla-sin":              {"n_layers": 12, "d_ff": 4 * 768, "attention_type": "mla", "pos_emb_type": "sinusoidal", "use_moe": False, "use_conv": False},
    "gqa-rope-swiglu-moe":  {"n_layers": 12, "d_ff": 1408,    "attention_type": "gqa", "pos_emb_type": "rope",       "use_moe": True,  "use_conv": False},
    "swiglu-moe-rope-conv": {"n_layers": 12, "d_ff": 1408,    "attention_type": "gqa", "pos_emb_type": "rope",       "use_moe": True,  "use_conv": True, "local_kernel_size": 25},
}
MAIN_MODEL = "gqa-rope-swiglu-moe"   # model used for the technique / ablation experiments

# ── Sequence layout:  <s> [document tokens] [SEP] [summary tokens] </s>  (all inside TRAIN_BLOCK_SIZE) ──
SEPARATOR_TEXT     = "\n\nSummary:\n"
MAX_SUMMARY_TOKENS = 256          # reference summaries are cut to this; also the cap for generation (check the EDA percentiles!)
RESERVED_PREFIX    = 16           # positions reserved for soft prompt / prefix tokens

# ── Long-document input strategies ──
HEAD_TAIL_SPLIT = 0.6             # head_tail: fraction of the budget taken from the beginning
UNIFORM_BLOCKS  = 8               # uniform: number of evenly spaced windows
ANCHOR_FRAC     = 0.2             # sentence_select: budget fraction always taken from the start AND from the end

# ── Hierarchical (chunk summaries -> summary of summaries) inference ──
HIER_SEGMENT_TOKENS         = 1024
HIER_SEGMENT_SUMMARY_TOKENS = 96
HIER_MAX_SEGMENTS           = 12

# ── Validation split carved out of train.json ──
VAL_FRACTION, VAL_MIN, VAL_MAX = 0.10, 50, 300
N_VAL_GEN = 100                   # validation documents that are actually summarised and scored with ROUGE/BERTScore

# ── Training ──
BATCH_DOCS      = 2
GRAD_ACCUM      = 8               # effective batch = BATCH_DOCS * GRAD_ACCUM
EVAL_BATCH_DOCS = 4
GEN_BATCH_DOCS  = 8
GRAD_CKPT       = True            # recompute activations in backward -> much lower memory
WARMUP_FRAC     = 0.06
WEIGHT_DECAY    = 0.01
ADALORA_ORTH_COEF = 0.01

SEED = 42
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## 2. Tokenizer

In [4]:
class ByteLevelBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.reserved = ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos = list(self.reserved); self.stoi = {}; self.merges = []; self.ranks = {}; self.merged_map = {}; self._cache = {}
        # Regex pattern matching words, numbers, individual punctuation, and whitespace sequences
        self.pattern = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[^\s\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+"""
        self.pat = re.compile(self.pattern, re.UNICODE)

    def fit(self, texts, max_unique_words=200000):
        """Train a byte-level BPE tokenizer using inverted indexing for fast merging."""
        raw_counts = Counter()
        iterator = [texts] if isinstance(texts, str) else texts

        # Preserve whitespace tokens when populating initial raw_counts
        for text in tqdm(iterator, desc="Building word vocabulary"):
            for chunk in re.findall(self.pattern, str(text), flags=re.UNICODE):
                raw_counts[chunk] += 1

        if max_unique_words and len(raw_counts) > max_unique_words:
            most_common = raw_counts.most_common(max_unique_words)
            raw_counts = dict(most_common)

        word_freqs = {}; pair_counts = defaultdict(int); pair_to_words = defaultdict(set)
        for word, count in raw_counts.items():
            encoded = word.encode("utf-8")
            byte_word = tuple(encoded[i:i+1] for i in range(len(encoded)))
            word_freqs[byte_word] = count

            for i in range(len(byte_word) - 1):
                pair = (byte_word[i], byte_word[i + 1])
                pair_counts[pair] += count
                pair_to_words[pair].add(byte_word)

        base_bytes = sorted({token for word in word_freqs for token in word})
        self.itos = list(self.reserved) + base_bytes; self.merges = []
        target_merges = self.vocab_size - len(self.itos)

        with tqdm(total=target_merges, desc="Training BPE Merges") as pbar:
            while len(self.itos) < self.vocab_size:
                if not pair_counts:
                    break
                best_pair = max(pair_counts, key=lambda p: (pair_counts[p], p[0], p[1]))
                if pair_counts[best_pair] <= 0:
                    break
                merged_token = best_pair[0] + best_pair[1]
                self.merges.append((best_pair, merged_token))
                self.itos.append(merged_token)
                
                words_to_update = list(pair_to_words[best_pair])
                del pair_counts[best_pair]
                del pair_to_words[best_pair]

                p0, p1 = best_pair
                for word in words_to_update:
                    count = word_freqs.get(word, 0)
                    if count == 0:
                        continue

                    new_word = []; i = 0; n = len(word)
                    while i < n:
                        if i + 1 < n and word[i] == p0 and word[i + 1] == p1:
                            new_word.append(merged_token)
                            i += 2
                        else:
                            new_word.append(word[i])
                            i += 1
                    new_word = tuple(new_word)
                    
                    for j in range(len(word) - 1):
                        old_p = (word[j], word[j + 1])
                        if old_p in pair_counts:
                            pair_counts[old_p] -= count
                            if pair_counts[old_p] <= 0:
                                del pair_counts[old_p]

                    for j in range(len(new_word) - 1):
                        new_p = (new_word[j], new_word[j + 1])
                        pair_counts[new_p] += count
                        pair_to_words[new_p].add(new_word)

                    del word_freqs[word]
                    word_freqs[new_word] = count

                pbar.update(1)

        self.vocab_size = len(self.itos)
        self.stoi = {token: idx for idx, token in enumerate(self.itos)}
        self.ranks = {pair: rank for rank, (pair, _) in enumerate(self.merges)}
        self.merged_map = {pair: merged for pair, merged in self.merges}
        self._cache = {}
        return self

    def encode_word(self, word):
        """Encode single word using memoization and local pair updates."""
        if word in self._cache:
            return self._cache[word]

        encoded = word.encode("utf-8")
        tokens = [encoded[i:i+1] for i in range(len(encoded))]

        if len(tokens) <= 1:
            res = [self.stoi.get(token, self.stoi["<unk>"]) for token in tokens]
            self._cache[word] = res
            return res

        while len(tokens) > 1:
            min_rank = float("inf")
            min_pair = None

            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                rank = self.ranks.get(pair, float("inf"))
                if rank < min_rank:
                    min_rank = rank
                    min_pair = pair

            if min_pair is None or min_rank == float("inf"):
                break

            merged = self.merged_map[min_pair]
            new_tokens = []
            i = 0
            n = len(tokens)
            p0, p1 = min_pair

            while i < n:
                if i + 1 < n and tokens[i] == p0 and tokens[i + 1] == p1:
                    new_tokens.append(merged)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens

        res = [self.stoi.get(token, self.stoi["<unk>"]) for token in tokens]
        self._cache[word] = res
        return res

    def encode(self, text):
        """Encode a sentence while preserving original spaces and newlines."""
        ids = [self.stoi["<s>"]]
        pattern = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[^\s\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+"""
        for chunk in re.findall(pattern, str(text), flags=re.UNICODE):
            ids.extend(self.encode_word(chunk))
        ids.append(self.stoi["</s>"])
        return ids
    
    def decode(self, ids):
        """Decode BPE token IDs back into text seamlessly."""
        byte_list = []
        for idx in ids:
            tok = self.itos[idx] if 0 <= idx < len(self.itos) else "<unk>"
            if tok not in self.reserved and isinstance(tok, bytes):
                byte_list.append(tok)
        try:
            return b"".join(byte_list).decode("utf-8", errors="ignore")
        except Exception:
            return ""

    def get_vocab_size(self):
        return self.vocab_size

def train_tokenizer(train_text: str, vocab_size: int):
    """Train A-1 BPE tokenizer on `train_text` and return the trained tokenizer."""
    tokenizer = ByteLevelBPE(vocab_size)
    tokenizer.fit(train_text)
    return tokenizer
    
# Load Pretrained Tokenizer
with open("tokenizerv3.pkl", "rb") as f:
    tokenizer = pickle.load(f)
BOS, PAD, UNK, EOS = (tokenizer.stoi[t] for t in ("<s>", "<pad>", "<unk>", "</s>"))
assert len(tokenizer.itos) == VOCAB_SIZE
print("tokenizer vocab:", len(tokenizer.itos))

tokenizer vocab: 20000


## 3. Model Class Definitions

In [5]:
@dataclass
class LMConfig:
    vocab_size: int = VOCAB_SIZE
    d_model: int = N_EMBD
    n_layers: int = N_LAYER
    n_heads: int = N_HEAD
    n_kv_heads: int = N_KV_HEAD
    kv_latent_dim: int = KV_LATENT_DIM
    d_ff: int = 1408          # For SwiGLU (2/3) * 4 * d_model => 2/3 * 2048 = 1365 --> round to nearest upper multiple of 64 (1408)
    max_seq_len: int  = TRAIN_BLOCK_SIZE
    dropout: float = 0.0            # ~1 pass over the data -> no need for dropout
    pos_emb_type: str = "rope"      # "rope" or "sinusoidal"
    attention_scale: float = None
    norm_first: bool = True         # FIX: pre-LN (post-LN + lr 1e-3 + no warm-up is unstable)
    n_experts:  int   = 4
    top_k:      int   = 2
    aux_loss_coef: float = 0.01
    attention_type: str = "gqa"
    use_moe: bool = False, 
    use_conv: bool = False
    local_kernel_size: int = 25
    
# Rotary Positional Embeddings
def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """ Returns (cos, sin) : the split-half rotate_half convention: channel i is paired with channel i+d_head/2. """
    inv = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    ang = torch.cat([torch.outer(torch.arange(max_seq_len).float(), inv)] * 2, -1)
    return ang.cos(), ang.sin()
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree rotation step used by the "rotate_half" formulation of RoPE."""
    x1, x2 = x[ ..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 : ]
    return torch.cat((-x2, x1), dim = -1)
def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """ x: (B, n_heads, T, d_head) -- a query or key tensor. cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T. Returns the rotated tensor, same shape as x. Formula: x_rotated = x * cos + rotate_half(x) * sin """
    cos, sin = (cos[None, None], sin[None, None]) if cos.dim() == 2 else (cos[:, None], sin[:, None])
    return x.float() * cos + rotate_half(x.float()) * sin

def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
    i = torch.arange(0, d_model, 2, dtype=torch.float32)
    angle = pos / (10000 ** (i / d_model))
    return torch.stack([torch.sin(angle), torch.cos(angle)], dim=-1).reshape(max_len, d_model)

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=True, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep, self.d_head = n_heads // n_kv_heads, d_model // n_heads
        self.use_rope = use_rope
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, position_ids=None, attn_mask=None, kv_cache=None, use_cache=False):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            q = apply_rope(q, self.rope_cos[position_ids], self.rope_sin[position_ids])
            k = apply_rope(k, self.rope_cos[position_ids], self.rope_sin[position_ids])
             
        # Append to KV cache if provided
        if kv_cache is not None:
            past_k, past_v = kv_cache
            k = torch.cat([past_k, k], dim=-2)
            v = torch.cat([past_v, v], dim=-2)
        new_kv_cache = (k, v) if use_cache else None

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # Only set is_causal=True if processing multiple prompt tokens at once (T > 1)
        is_causal = (attn_mask is None) and (kv_cache is None) and (T > 1)
        context = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.attn_dropout.p if self.training else 0.0,  is_causal=is_causal, scale=self.attention_scale)
        out = self.resid_dropout(self.out_proj(context.transpose(1, 2).reshape(B, T, C)))
        
        return (out, new_kv_cache) if use_cache else out

class MultiHeadLatentAttention(nn.Module):
    """Joint KV Compression Multi-Head Latent Attention module drop-in replacement for CausalSelfAttention."""

    def __init__(self, d_model, n_heads, kv_latent_dim, max_seq_len, dropout=0.0, attention_scale=None):
        super().__init__()
        assert (d_model % n_heads == 0 and 0 < kv_latent_dim < d_model), "d_model must be divisible by n_heads, and 0 < kv_latent_dim < d_model"

        self.n_heads, self.kv_latent_dim, self.d_head = n_heads, kv_latent_dim, d_model // n_heads
        self.attention_scale = ((1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale))

        # Projections
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_down_proj = nn.Linear(d_model, kv_latent_dim, bias=False)
        self.k_up_proj = nn.Linear(kv_latent_dim, d_model, bias=False)
        self.v_up_proj = nn.Linear(kv_latent_dim, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        # Dropouts
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, position_ids=None, attn_mask=None, kv_cache=None, use_cache=False):
        """Forward pass supporting sequence generation with KV cache and FlashAttention/SDPA."""
        B, T, C = x.shape
        q = (self.q_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2))  # (B, n_heads, T, d_head)
        c_kv = self.kv_down_proj(x)  # (B, T, kv_latent_dim)

        if kv_cache is not None:
            past_c_kv = (kv_cache[0] if isinstance(kv_cache, (tuple, list)) else kv_cache)
            c_kv = torch.cat([past_c_kv, c_kv], dim=1)  # Append along sequence dim T
        new_kv_cache = (c_kv,) if use_cache else None

        k = (self.k_up_proj(c_kv).view(B, -1, self.n_heads, self.d_head).transpose(1, 2))  # (B, n_heads, T_total, d_head)
        v = (self.v_up_proj(c_kv).view(B, -1, self.n_heads, self.d_head).transpose(1, 2))  # (B, n_heads, T_total, d_head)
        is_causal = (attn_mask is None) and (kv_cache is None) and (T > 1)
        
        context = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=is_causal, scale=self.attention_scale)
        out = context.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(out))

        return (out, new_kv_cache) if use_cache else out

### Model Definitions
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)
        
class SwiGLUFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False); self.up   = nn.Linear(d_model, d_ff, bias=False); self.down = nn.Linear(d_ff, d_model, bias=False); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

class TopKRouter(nn.Module):
    def __init__(self, d_model: int, n_experts: int, top_k: int):
        super().__init__()
        self.top_k  = top_k; self.linear = nn.Linear(d_model, n_experts, bias=False)
    def forward(self, x_flat):                           
        logits = self.linear(x_flat); topk_vals, topk_idx = torch.topk(logits, self.top_k, dim=-1)
        weights = F.softmax(topk_vals, dim=-1)            # (B*T, top_k)
        dispatch = torch.zeros_like(logits, dtype=weights.dtype)
        dispatch.scatter_(-1, topk_idx, weights)          # (B*T, n_experts)
        return dispatch, logits                           # logits kept for aux loss

class MoEFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, n_experts, top_k, dropout, aux_coef=0.01):
        super().__init__()
        self.router   = TopKRouter(d_model, n_experts, top_k)
        self.experts  = nn.ModuleList([SwiGLUFeedForward(d_model, d_ff, dropout) for _ in range(n_experts)])
        self.aux_coef = aux_coef; self.aux_loss = torch.tensor(0.0)

    def forward(self, x):
        B, T, D = x.shape
        x_flat = x.reshape(B * T, D)
        dispatch, router_logits = self.router(x_flat)

        z = torch.logsumexp(router_logits, dim=-1)
        self.aux_loss = self.aux_coef * (z ** 2).mean()
        out = torch.zeros_like(x_flat)
        for e_idx, expert in enumerate(self.experts):
            w  = dispatch[:, e_idx];  mask = w > 0
            if not mask.any():
                continue
            out[mask] += w[mask].unsqueeze(-1) * expert(x_flat[mask])

        return out.reshape(B, T, D)

class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.norm_first = cfg.norm_first; self.ln1 = nn.LayerNorm(cfg.d_model); self.ln2 = nn.LayerNorm(cfg.d_model)
        if(cfg.attention_type == "mla"):
            self.attn = MultiHeadLatentAttention(cfg.d_model, cfg.n_heads, cfg.kv_latent_dim, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        else:
            self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len, cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"), attention_scale=cfg.attention_scale)
        if(cfg.use_moe):
            self.ffn  = MoEFeedForward(cfg.d_model, cfg.d_ff, cfg.n_experts, cfg.top_k, cfg.dropout, cfg.aux_loss_coef)
        else:
            self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        
    def _attn(self, h, position_ids, attn_mask, kv_cache, use_cache):
        r = self.attn(h, position_ids=position_ids, attn_mask=attn_mask, kv_cache=kv_cache, use_cache=use_cache)
        return r if use_cache else (r, None)
        
    def forward(self, x, position_ids, attn_mask=None, kv_cache=None, use_cache=False):
        if self.norm_first:
            a, new_cache = self._attn(self.ln1(x), position_ids, attn_mask, kv_cache, use_cache)
            x = x + a
            x = x + self.ffn(self.ln2(x))
        else:
            a, new_cache = self._attn(x, position_ids, attn_mask, kv_cache, use_cache)
            x = self.ln1(x + a)
            x = self.ln2(x + self.ffn(x))
        return x, new_cache
        
class LocalDepthwiseConv(nn.Module):
    """ Depthwise causal convolution over the sequence dimension. Acts on each channel independently — O(T * C * kernel) cost. kernel_size = 25 covers ~25 tokens of local context. """
    def __init__(self, d_model: int, kernel_size: int = 25, dropout: float = 0.0):
        super().__init__()
        self.kernel_size = kernel_size
        self.conv = nn.Conv1d(in_channels=d_model, out_channels=d_model, kernel_size=kernel_size, padding=0, groups=d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
        nn.init.normal_(self.conv.weight, 0.0, 0.02)
        nn.init.normal_(self.proj.weight, 0.0, 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, C) → (B, T, C)"""
        B, T, C = x.shape
        xc = x.transpose(1, 2)
        xc = F.pad(xc, (self.kernel_size - 1, 0))
        xc = self.conv(xc)          # (B, C, T)  — output length = T (causal)
        xc = xc.transpose(1, 2)    # (B, T, C)
        return self.drop(self.proj(F.gelu(xc)))

class ParallelLocalGlobalBlock(nn.Module):
    """Pre-LN block with global attention and local conv running IN PARALLEL."""
    def __init__(self, cfg):
        super().__init__()
        self.ln1  = nn.RMSNorm(cfg.d_model); self.ln2  = nn.RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len, cfg.dropout, use_rope=cfg.pos_emb_type == "rope", attention_scale=cfg.attention_scale)
        self.conv = LocalDepthwiseConv(cfg.d_model, kernel_size=cfg.local_kernel_size, dropout=cfg.dropout)
        self.ffn  = MoEFeedForward(cfg.d_model, cfg.d_ff, cfg.n_experts, cfg.top_k, cfg.dropout, cfg.aux_loss_coef)
        self.alpha = nn.Parameter(torch.ones(1))    # global attention scale
        self.beta  = nn.Parameter(torch.full((1,), 0.1))  # local conv scale

    def forward(self, x, position_ids, attn_mask=None, kv_cache=None, use_cache=False):
        normed = self.ln1(x)
        attn_out = self.attn(normed, position_ids=position_ids, attn_mask=attn_mask, kv_cache=kv_cache, use_cache=use_cache)
        attn_out, new_cache = attn_out if use_cache else attn_out, None
        conv_out = self.conv(normed)
        
        x = x + self.alpha * attn_out + self.beta * conv_out
        x = x + self.ffn(self.ln2(x))
        return x, new_cache

    @property
    def aux_loss(self):
        return self.ffn.aux_loss


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.use_sin = cfg.pos_emb_type == "sinusoidal"
        self.emb_scale = math.sqrt(cfg.d_model) if self.use_sin else 1.0
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if self.use_sin:
            self.register_buffer("pos_emb", sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model), persistent=False)
        if cfg.use_conv:
            self.blocks = nn.ModuleList([ParallelLocalGlobalBlock(cfg) for _ in range(cfg.n_layers)])
        else:
            self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self._init_weights()
        self.head.weight = self.tok_emb.weight # tie AFTER init so the embedding init survives

    def aux_loss(self):
        return sum(blk.ffn.aux_loss for blk in self.blocks)
        
    def _init_weights(self):
        emb_std = self.cfg.d_model ** -0.5 if self.use_sin else 0.02
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, mean=0.0, std=emb_std)
        for n, p in self.named_parameters():
            if n.endswith("attn.out_proj.weight") or n.endswith("ffn.net.2.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * self.cfg.n_layers))
    
    def forward(self, idx: torch.Tensor, kv_cache_list=None, use_cache=False, attention_mask=None) -> torch.Tensor:
        """attention_mask: optional (B, past+T) tensor, 1 = real token, 0 = left padding."""
        B, T = idx.shape
        past = kv_cache_list[0][0].shape[-2] if kv_cache_list is not None else 0
        S = past + T

        if attention_mask is None:
            position_ids = torch.arange(past, S, device=idx.device).unsqueeze(0).expand(B, T)
            attn_mask = None                                   # fast path: is_causal=True inside SDPA
        else:
            position_ids = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)[:, past:S]
            q_idx = torch.arange(past, S, device=idx.device)[:, None]
            k_idx = torch.arange(S, device=idx.device)[None, :]
            allowed = (k_idx <= q_idx)[None] & attention_mask[:, None, :S].bool()   # causal & not-padding
            allowed = allowed | (k_idx == q_idx)[None]         # pad rows attend to themselves (no NaNs)
            attn_mask = allowed.unsqueeze(1)                   # (B, 1, T, S)

        x = self.tok_emb(idx) * self.emb_scale
        if self.use_sin:
            x = x + self.pos_emb[position_ids]

        new_caches = []
        for i, block in enumerate(self.blocks):
            layer_cache = kv_cache_list[i] if kv_cache_list is not None else None
            x, c = block(x, position_ids, attn_mask, layer_cache, use_cache)
            new_caches.append(c)

        logits = self.head(self.ln_f(x))
        return (logits, new_caches) if use_cache else logits
        
    @torch.no_grad()
    def generate(self, input_ids, attention_mask=None, max_new_tokens=384, temperature=0.7, top_p=0.9,
                 repetition_penalty=1.0, banned_token_ids=(0, 1, 2, 3), do_sample=True):
        self.eval()
        out = input_ids
        if attention_mask is None:
            attention_mask = torch.ones_like(out)
    
        logits, kv = self(out, use_cache=True, attention_mask=attention_mask)      # prefill
        next_logits = logits[:, -1, :].float()
    
        for step in range(max_new_tokens):
            l = next_logits.clone()
            if repetition_penalty != 1.0:
                sc = l.gather(1, out); sc = torch.where(sc < 0, sc * repetition_penalty, sc / repetition_penalty)
                l = l.scatter(1, out, sc)
            if banned_token_ids:                       # never emit <pad>/<unk>/<s>/</s>: we need 128 words
                l[:, list(banned_token_ids)] = float("-inf")

            if do_sample and temperature > 0.0:
                l = l / temperature
                if top_p < 1.0:
                    sorted_logits, sorted_idx = torch.sort(l, descending=True)
                    cum = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                    remove = cum > top_p
                    remove[..., 1:] = remove[..., :-1].clone()
                    remove[..., 0] = False
                    l = l.masked_fill(remove.scatter(1, sorted_idx, remove), float("-inf"))
                next_token = torch.multinomial(F.softmax(l, dim=-1), num_samples=1)
            else:
                next_token = l.argmax(dim=-1, keepdim=True)

            out = torch.cat([out, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)
            if step == max_new_tokens - 1:
                break
            logits, kv = self(next_token, kv_cache_list=kv, use_cache=True, attention_mask=attention_mask)
            next_logits = logits[:, -1, :].float()
        return out

## 4. Data Loading

In [6]:
def dapt_path(model_name):
    return f"{OUT}/dapt_{model_name}.pt"

def load_phase1_model(model_name):
    cfg = LMConfig()
    for attr, val in PHASE1_CONFIGS[model_name].items():
        setattr(cfg, attr, val)

    checkpoint_data = torch.load(f"./base-models/{model_name}.pt", map_location="cpu", weights_only=False)
    base_lm = DecoderLM(cfg)
    base_lm.load_state_dict(checkpoint_data["model"], strict=True)
    del checkpoint_data; gc.collect()
    return base_lm

def load_base(model_name, init_from):
    """init_from: 'phase1' = the Phase 1 weights; 'dapt' = Phase 1 weights after domain-adaptive pretraining (Cell 16)."""
    base_lm = load_phase1_model(model_name)
    if init_from == "dapt":
        base_lm.load_state_dict(torch.load(dapt_path(model_name), map_location="cpu", weights_only=False), strict=True)
    else:
        assert init_from == "phase1", init_from
    return base_lm

@dataclass
class DocSplit:
    ids: list
    texts: list              # cleaned documents
    fingerprints: list       # md5 of cleaned document (dedup + token cache key)
    summaries: list          # cleaned reference summaries (None for test)
    doc_tokens: list         # np.int32 array per case: document tokens WITHOUT <s>/</s> (filled in Cell 7)
    piece_ends: list         # np.int32 array per case: token offset where each text piece ends (Cell 7)
    summary_tokens: list     # np.int32 array per case (None for test; Cell 7)

def clean_document(text):
    """Light cleanup. Keep consistent with whatever was applied to the Phase 1 pretraining text."""
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)          # collapse runs of spaces/tabs
    text = re.sub(r"\n{3,}", "\n\n", text)       # collapse many blank lines
    return text.strip()

def clean_summary(text):
    """Summaries are generated as one paragraph, so references are normalised to one paragraph too."""
    return re.sub(r"\s+", " ", text).strip()

def fingerprint(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def read_split(relative_path):
    path = hf_hub_download(repo_id=HF_DATASET_REPO, repo_type="dataset", filename=f"{PHASE6_DIR}/{relative_path}")
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)
    assert isinstance(records, list), f"{relative_path}: expected a list of records, got {type(records)}"
    return records

def build_split(records, has_summary, split_name):
    ids = [r[ID_KEY] for r in records]
    assert len(set(ids)) == len(ids), f"{split_name}: duplicate case ids"
    assert all(isinstance(r[DOC_KEY], str) for r in records), f"{split_name}: non-string document"
    texts = [clean_document(r[DOC_KEY]) for r in records]
    assert all(len(t) > 0 for t in texts), f"{split_name}: empty document after cleaning"
    summaries = None
    if has_summary:
        assert all(isinstance(r[SUMMARY_KEY], str) for r in records), f"{split_name}: non-string summary"
        summaries = [clean_summary(r[SUMMARY_KEY]) for r in records]
        # assert all(len(s) > 0 for s in summaries), f"{split_name}: empty summary after cleaning"
    return DocSplit(ids, texts, [fingerprint(t) for t in texts], summaries, None, None, None)

def select_rows(split, rows):
    rows = list(rows)
    pick = lambda values: None if values is None else [values[i] for i in rows]
    return DocSplit(pick(split.ids), pick(split.texts), pick(split.fingerprints), pick(split.summaries),
                    pick(split.doc_tokens), pick(split.piece_ends), pick(split.summary_tokens))

train_all  = build_split(read_split("Train/train.json"), True,  "train")
# 0) drop train cases whose reference summary is empty after cleaning (nothing to learn from)
non_empty_rows = [i for i, summary in enumerate(train_all.summaries) if len(summary) > 0]
print(f"train: removed {len(train_all.ids) - len(non_empty_rows)} cases with an empty reference summary")
train_all = select_rows(train_all, non_empty_rows)

test_split = build_split(read_split("Test/test.json"),   False, "test")
print("raw sizes -> train", len(train_all.ids), "| test", len(test_split.ids))

# 1) identical documents inside train: keep the first occurrence (a duplicate in train AND val would leak)
first_row, keep_rows, summary_conflicts = {}, [], 0
for row, fp in enumerate(train_all.fingerprints):
    if fp in first_row:
        summary_conflicts += int(train_all.summaries[first_row[fp]] != train_all.summaries[row])
    else:
        first_row[fp] = row; keep_rows.append(row)
print(f"train: removed {len(train_all.ids) - len(keep_rows)} duplicate documents ({summary_conflicts} with different summaries)")
train_all = select_rows(train_all, keep_rows)

# 2) held-out validation split (no public dev set exists)
rng = np.random.RandomState(SEED)
permutation = rng.permutation(len(train_all.ids))
val_size = int(np.clip(int(VAL_FRACTION * len(permutation)), VAL_MIN, VAL_MAX))
assert len(permutation) > 4 * val_size, "dataset too small for the configured validation split"
val_split   = select_rows(train_all, permutation[:val_size])
train_split = select_rows(train_all, permutation[val_size:])

# 3) test is never filtered (one summary per test id is required); only report overlap
train_fps = set(train_all.fingerprints)
print("test documents also present in train:", sum(fp in train_fps for fp in test_split.fingerprints))
print("id overlap train/test:", len(set(train_all.ids) & set(test_split.ids)))

if SMOKE_TEST:
    smoke_rng = np.random.RandomState(SEED)
    train_split = select_rows(train_split, smoke_rng.choice(len(train_split.ids), 48, replace=False))
    val_split   = select_rows(val_split,   smoke_rng.choice(len(val_split.ids),    8, replace=False))
    test_split  = select_rows(test_split,  smoke_rng.choice(len(test_split.ids),   8, replace=False))
print("final sizes -> train", len(train_split.ids), "| val", len(val_split.ids), "| test", len(test_split.ids))

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

train: removed 2 cases with an empty reference summary


test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

raw sizes -> train 7028 | test 100
train: removed 27 duplicate documents (0 with different summaries)
test documents also present in train: 0
id overlap train/test: 0
final sizes -> train 6701 | val 300 | test 100


In [7]:
# A document is cut into "pieces" at sentence ends and line breaks. Pieces start with their leading whitespace, so
# tokenising piece by piece matches tokenising the whole text, and joining pieces gives the original text exactly.
PIECE_BOUNDARY = re.compile(r"(?<=[.!?;:])(?=\s)|(?=\n)")
def split_pieces(text):
    return [p for p in PIECE_BOUNDARY.split(text) if p]

TOKEN_CACHE_PATH = f"{OUT}/token_cache_tokenizerv3_{VOCAB_SIZE}.pkl"
token_cache = pickle.load(open(TOKEN_CACHE_PATH, "rb")) if os.path.exists(TOKEN_CACHE_PATH) else {}

def tokenize_document(text):
    """-> (flat int32 token array without <s>/</s>, int32 array with the end offset of every piece)."""
    per_piece = [tokenizer.encode(piece)[1:-1] for piece in split_pieces(text)]
    ends = np.cumsum([len(tokens) for tokens in per_piece]).astype(np.int32)
    flat = np.fromiter(itertools.chain.from_iterable(per_piece), dtype=np.int32, count=int(ends[-1]))
    return flat, ends

assert "".join(split_pieces(train_split.texts[0])) == train_split.texts[0], "piece splitting must be lossless"

for split_name, split in [("train", train_split), ("val", val_split), ("test", test_split)]:
    for text, fp in tqdm(zip(split.texts, split.fingerprints), total=len(split.texts), desc=f"tokenize {split_name}"):
        if fp not in token_cache:
            token_cache[fp] = tokenize_document(text)
    split.doc_tokens = [token_cache[fp][0] for fp in split.fingerprints]
    split.piece_ends = [token_cache[fp][1] for fp in split.fingerprints]
    if split.summaries is not None:
        split.summary_tokens = [np.array(tokenizer.encode(s)[1:-1], dtype=np.int32) for s in split.summaries]
        assert all(len(t) > 0 for t in split.summary_tokens)
pickle.dump(token_cache, open(TOKEN_CACHE_PATH, "wb"))

# prompt pieces and the input budget for the document part
SEP_IDS = tokenizer.encode(SEPARATOR_TEXT)[1:-1]
NL_IDS  = tokenizer.encode("\n")[1:-1]
DOC_BUDGET = TRAIN_BLOCK_SIZE - RESERVED_PREFIX - 1 - len(SEP_IDS) - MAX_SUMMARY_TOKENS - 1   # <s> doc SEP summary </s>
print(f"separator ids {SEP_IDS} | document token budget {DOC_BUDGET}")

# sanity: byte-level BPE should almost never emit <unk>
all_tokens = np.concatenate(train_split.doc_tokens)
print("unk fraction in train tokens:", (all_tokens == UNK).mean())

WORD_RE = re.compile(r"[a-z]{3,}")

def build_idf(texts):
    doc_freq = Counter()
    for text in texts:
        doc_freq.update(set(WORD_RE.findall(text.lower())))
    return {word: math.log((len(texts) + 1) / (count + 1)) + 1.0 for word, count in doc_freq.items()}

IDF = build_idf(train_split.texts)          # statistics from train documents only

def select_important_pieces(tokens, piece_ends, text, budget):
    """Importance-based context selection: keep the start and end of the case, then fill the budget with the
    sentence pieces whose words are most characteristic (tf-idf) for this case; original order is preserved."""
    pieces = split_pieces(text)
    assert len(pieces) == len(piece_ends), "text pieces and cached token pieces disagree"
    starts = np.concatenate([[0], piece_ends[:-1]])
    lengths = (piece_ends - starts).tolist()
    piece_words = [set(WORD_RE.findall(piece.lower())) for piece in pieces]
    pieces_with_word = Counter(w for words in piece_words for w in words)
    weight = {w: count * IDF[w] for w, count in pieces_with_word.items() if w in IDF}
    scores = [sum(weight[w] for w in words if w in weight) / math.sqrt(len(words) + 1) for words in piece_words]

    keep, used = set(), 0
    head_used = 0
    for i in range(len(pieces)):                                         # anchor: beginning
        if head_used + lengths[i] > ANCHOR_FRAC * budget: break
        keep.add(i); head_used += lengths[i]
    tail_used = 0
    for i in reversed(range(len(pieces))):                               # anchor: end
        if i in keep or tail_used + lengths[i] > ANCHOR_FRAC * budget: break
        keep.add(i); tail_used += lengths[i]
    used = head_used + tail_used
    for i in np.argsort(scores)[::-1]:                                   # fill with the most important pieces
        i = int(i)
        if i not in keep and used + lengths[i] <= budget:
            keep.add(i); used += lengths[i]
    assert used <= budget
    return np.concatenate([tokens[starts[i]: piece_ends[i]] for i in sorted(keep)])

def select_doc_tokens(tokens, piece_ends, text, strategy, budget):
    """Which part of a (possibly very long) case is shown to the model."""
    n = len(tokens)
    if n <= budget:
        return tokens
    if strategy == "head":
        return tokens[:budget]
    if strategy == "tail":
        return tokens[-budget:]
    if strategy == "head_tail":                                          # facts at the start + decision at the end
        n_head = int(budget * HEAD_TAIL_SPLIT)
        return np.concatenate([tokens[:n_head], tokens[-(budget - n_head):]])
    if strategy == "uniform":                                            # evenly spaced windows over the whole case
        window = budget // UNIFORM_BLOCKS
        starts = np.linspace(0, n - window, UNIFORM_BLOCKS).round().astype(int)
        return np.concatenate([tokens[s: s + window] for s in starts])
    if strategy == "sentence_select":
        return select_important_pieces(tokens, piece_ends, text, budget)
    raise ValueError(f"unknown strategy: {strategy}")

DOC_INPUT_CACHE = {}
def get_doc_inputs(split_name, split, strategy):
    key = (split_name, strategy, len(split.ids))
    if key not in DOC_INPUT_CACHE:
        DOC_INPUT_CACHE[key] = [select_doc_tokens(tokens, ends, text, strategy, DOC_BUDGET)
                                for tokens, ends, text in tqdm(zip(split.doc_tokens, split.piece_ends, split.texts),
                                                               total=len(split.ids), desc=f"inputs {split_name}/{strategy}", leave=False)]
    return DOC_INPUT_CACHE[key]

def build_prompt(doc_tokens):
    return [BOS] + doc_tokens.tolist() + SEP_IDS

class SummaryDataset(Dataset):
    """Training item: ids = prompt + summary + </s>, labels = -100 on the prompt, token ids on summary + </s>."""
    def __init__(self, split, doc_inputs):
        self.split, self.doc_inputs = split, doc_inputs
    def __len__(self):
        return len(self.split.ids)
    def __getitem__(self, i):
        prompt = build_prompt(self.doc_inputs[i])
        summary = self.split.summary_tokens[i][:MAX_SUMMARY_TOKENS].tolist() + [EOS]
        ids = prompt + summary
        assert len(ids) <= TRAIN_BLOCK_SIZE - RESERVED_PREFIX
        return {"ids": ids, "labels": [-100] * len(prompt) + summary}

def collate_lm(items):
    """Right-pad: causal attention means padding never influences real tokens."""
    max_len = max(len(it["ids"]) for it in items)
    ids = torch.full((len(items), max_len), PAD, dtype=torch.long)
    labels = torch.full((len(items), max_len), -100, dtype=torch.long)
    for b, it in enumerate(items):
        ids[b, :len(it["ids"])] = torch.tensor(it["ids"])
        labels[b, :len(it["labels"])] = torch.tensor(it["labels"])
    return {"ids": ids, "labels": labels}

tokenize test: 100%|██████████| 100/100 [00:00<00:00, 621378.37it/s]


separator ids [4, 4, 56, 359, 81, 394, 31, 4] | document token budget 1766
unk fraction in train tokens: 1.8724694669360588e-05


In [8]:
doc_len = np.array([len(t) for t in train_split.doc_tokens])
sum_len = np.array([len(t) for t in train_split.summary_tokens])
pct = [50, 75, 90, 95, 99, 100]
print(f"train n={len(train_split.ids)} | val n={len(val_split.ids)} | test n={len(test_split.ids)}")
print("document tokens percentiles (50/75/90/95/99/max):", np.percentile(doc_len, pct).astype(int))
print("summary  tokens percentiles (50/75/90/95/99/max):", np.percentile(sum_len, pct).astype(int))
print("test document tokens percentiles:", np.percentile([len(t) for t in test_split.doc_tokens], pct).astype(int))
print(f"documents longer than the input budget ({DOC_BUDGET}): {(doc_len > DOC_BUDGET).mean():.1%}")
print(f"summaries longer than MAX_SUMMARY_TOKENS ({MAX_SUMMARY_TOKENS}) and therefore cut: {(sum_len > MAX_SUMMARY_TOKENS).mean():.1%}")
print(f"compression: median doc/summary token ratio {np.median(doc_len / sum_len):.1f}")

# Where in a document do the summary's words come from? (content words found in each third of the document)
probe_rows = range(min(200, len(val_split.ids)))
thirds = []
for i in probe_rows:
    doc, summary_words = val_split.texts[i], set(WORD_RE.findall(val_split.summaries[i].lower()))
    n = len(doc)
    parts = [doc[:n // 3], doc[n // 3: 2 * n // 3], doc[2 * n // 3:]]
    thirds.append([len(summary_words & set(WORD_RE.findall(p.lower()))) / len(summary_words) for p in parts])
print("fraction of summary content words present in the first / middle / last third of the document:", np.mean(thirds, axis=0).round(3))

# Trivial lead baseline: first K words of the document (K = mean reference length). Anything trained should beat this.
scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
mean_words = int(np.mean([len(s.split()) for s in train_split.summaries]))
lead = [" ".join(val_split.texts[i].split()[:mean_words]) for i in probe_rows]
print(f"lead-{mean_words}-words baseline on val: ROUGE-1 {np.mean([scorer.score(val_split.summaries[i], p)['rouge1'].fmeasure for i, p in zip(probe_rows, lead)]):.4f}"
      f" | ROUGE-L {np.mean([scorer.score(val_split.summaries[i], p)['rougeL'].fmeasure for i, p in zip(probe_rows, lead)]):.4f}")

for i in range(2):
    print(f"\n--- id {train_split.ids[i]} | document start ---\n{train_split.texts[i][:400]}\n--- reference summary ---\n{train_split.summaries[i][:500]}")

train n=6701 | val n=300 | test n=100
document tokens percentiles (50/75/90/95/99/max): [  3703   5852   9188  13248  28913 165091]
summary  tokens percentiles (50/75/90/95/99/max): [  762  1158  1798  2435  5198 33191]
test document tokens percentiles: [ 3835  6480 10570 16401 34488 46337]
documents longer than the input budget (1766): 86.9%
summaries longer than MAX_SUMMARY_TOKENS (256) and therefore cut: 96.0%
compression: median doc/summary token ratio 4.5
fraction of summary content words present in the first / middle / last third of the document: [0.537 0.56  0.625]
lead-837-words baseline on val: ROUGE-1 0.5085 | ROUGE-L 0.2592

--- id 2908 | document start ---
Appeal No. 2433 of 1966. Appeal from the judgment and decree dated August 2, 1965 of the Bombay High Court, Nagpur Bench in Appeal No. 113 of 1959 from original decree. V. section Desai, V. N. Swamy, K. Rajendra Chaudhuri and K. R. Chaudhuri, for the appellants. M. N. Phadke and A. G. Ratnaparkhi, for the respondents. The

## 5. Finetuning Methods

In [9]:
class LoRALinear(nn.Module):
    """y = Wx + (alpha/r) * B(A x)   (additive low-rank update)."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

class LoRMALinear(nn.Module):
    """y = Wx + (alpha/r) * B(A(Wx))  ==  (I + s*BA) W x   (multiplicative low-rank update)."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(y, self.A), self.B)

class QLoRALinear(nn.Module):
    """Frozen weight stored as int8 (per-output-channel absmax quantisation, dequantised on the fly) + trainable LoRA.
    Weight-only 8-bit simulation of QLoRA (no bitsandbytes needed)."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        assert base.bias is None, "QLoRALinear expects bias-free projections"
        weight = base.weight.detach()
        scale = weight.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / 127.0
        self.register_buffer("w_int8", torch.round(weight / scale).clamp(-127, 127).to(torch.int8))
        self.register_buffer("w_scale", scale)
        self.lora_scale = alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        dequantised = self.w_int8.float() * self.w_scale
        return F.linear(x, dequantised) + self.lora_scale * F.linear(F.linear(x, self.A), self.B)

class AdaLoRALinear(nn.Module):
    """Delta W = P diag(lambda * mask) Q. Orthogonality penalty + rank pruning (see AdaLoRAController)."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base, self.rank, self.scale = base, rank, alpha / rank
        self.P    = nn.Parameter(torch.empty(base.out_features, rank))
        self.diag = nn.Parameter(torch.zeros(rank))
        self.Q    = nn.Parameter(torch.empty(rank, base.in_features))
        self.register_buffer("mask", torch.ones(rank))           # 0 = pruned singular value
        nn.init.orthogonal_(self.P); nn.init.orthogonal_(self.Q)
    def orth_loss(self):
        eye = torch.eye(self.rank, device=self.P.device)
        return ((self.P.T @ self.P - eye) ** 2).sum() + ((self.Q @ self.Q.T - eye) ** 2).sum()
    def forward(self, x):
        low_rank = F.linear(x, self.Q) * (self.diag * self.mask)
        return self.base(x) + self.scale * F.linear(low_rank, self.P)

class AdaLoRAController:
    """Prunes singular values by smoothed sensitivity |lambda * grad| following a cubic budget schedule."""
    def __init__(self, modules, total_steps, final_ratio, warmup_frac=0.1, freeze_frac=0.2, prune_every=20, beta=0.85):
        self.modules, self.prune_every, self.beta = modules, prune_every, beta
        self.initial_budget = sum(m.rank for m in modules)
        self.final_budget = int(self.initial_budget * final_ratio)
        self.start_step = int(total_steps * warmup_frac)
        self.end_step = int(total_steps * (1 - freeze_frac))
        self.importance = [torch.zeros_like(m.diag) for m in modules]
    def budget_at(self, step):
        if step < self.start_step: return self.initial_budget
        if step >= self.end_step:  return self.final_budget
        progress = (step - self.start_step) / (self.end_step - self.start_step)
        return int(self.final_budget + (self.initial_budget - self.final_budget) * (1 - progress) ** 3)
    @torch.no_grad()
    def update(self, step):
        """Call after backward (grads unscaled), before optimizer.step()."""
        if step >= self.end_step:
            return
        for module, imp in zip(self.modules, self.importance):
            imp.mul_(self.beta).add_((1 - self.beta) * (module.diag * module.diag.grad).abs())
        if step >= self.start_step and step % self.prune_every == 0:
            all_scores = torch.cat(self.importance)
            threshold = torch.topk(all_scores, self.budget_at(step)).values[-1]
            for module, imp in zip(self.modules, self.importance):
                module.mask.copy_((imp >= threshold).float())
    def active_rank(self):
        return int(sum(m.mask.sum().item() for m in self.modules))

class BottleneckAdapter(nn.Module):
    """h + up(gelu(down(h))), zero-initialised so training starts at the identity."""
    def __init__(self, d_model, bottleneck):
        super().__init__()
        self.down, self.up = nn.Linear(d_model, bottleneck), nn.Linear(bottleneck, d_model)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))

class AdaptedSublayer(nn.Module):
    """Applies an adapter to the output of an attention / FFN sub-layer (before the residual add).
    Passes (output, kv_cache) tuples through, so it also works with the KV cache."""
    def __init__(self, inner, d_model, bottleneck):
        super().__init__()
        self.inner, self.adapter = inner, BottleneckAdapter(d_model, bottleneck)
    def forward(self, *args, **kwargs):
        out = self.inner(*args, **kwargs)
        if isinstance(out, tuple):
            return self.adapter(out[0]), out[1]
        return self.adapter(out)

def add_adapters(backbone, d_model, bottleneck, style):
    """houlsby: adapters after attention AND after the FFN of every block. pfeiffer: only after the FFN."""
    for block in backbone.blocks:
        block.ffn = AdaptedSublayer(block.ffn, d_model, bottleneck)
        if style == "houlsby":
            block.attn = AdaptedSublayer(block.attn, d_model, bottleneck)

class SoftPrefix(nn.Module):
    """n learned vectors prepended to the embeddings of every sequence (backbone stays frozen).
    reparam=False -> prompt tuning; reparam=True -> prefix-style MLP reparameterisation of the vectors."""
    def __init__(self, n_tokens, dim, token_embeddings, reparam):
        super().__init__()
        init_rows = torch.randint(0, token_embeddings.size(0), (n_tokens,))
        self.latent = nn.Parameter(token_embeddings[init_rows].clone())
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.Tanh(), nn.Linear(dim, dim)) if reparam else None
    def forward(self):
        return self.latent if self.mlp is None else self.mlp(self.latent)

PEFT_WRAPPERS = {"lora": LoRALinear, "lorma": LoRMALinear, "qlora": QLoRALinear, "adalora": AdaLoRALinear}
ATTN_PROJ_NAMES = ("q_proj", "k_proj", "v_proj", "out_proj", "kv_down_proj", "k_up_proj", "v_up_proj")  # GQA and MLA names

def wrap_attention_projections(backbone, wrapper_cls, rank, alpha):
    """Wrap every attention projection that exists in this architecture (GQA / MLA / conv variants)."""
    for block in backbone.blocks:
        for name in ATTN_PROJ_NAMES:
            if hasattr(block.attn, name):
                setattr(block.attn, name, wrapper_cls(getattr(block.attn, name), rank, alpha))

In [10]:
class SummarizerLM(nn.Module):
    """The Phase 1 decoder (language-model head tied to the embeddings) + an optional soft prefix."""
    def __init__(self, backbone, soft_prefix):
        super().__init__()
        self.backbone, self.soft_prefix = backbone, soft_prefix
    @property
    def n_prefix(self):
        return 0 if self.soft_prefix is None else self.soft_prefix.latent.size(0)

def embed_tokens(model, ids, with_prefix):
    x = model.backbone.tok_emb(ids) * model.backbone.emb_scale
    if with_prefix and model.soft_prefix is not None:
        prefix = model.soft_prefix().unsqueeze(0).expand(ids.size(0), -1, -1).to(x.dtype)
        x = torch.cat([prefix, x], dim=1)
    return x

def build_attn_mask(attention_mask, past, T):
    """Causal mask that also hides left padding. attention_mask: [B, past+T] (True = real token)."""
    total = past + T
    q_idx = torch.arange(past, total, device=attention_mask.device)[:, None]
    k_idx = torch.arange(total, device=attention_mask.device)[None, :]
    allowed = (k_idx <= q_idx)[None] & attention_mask[:, None, :total].bool()
    allowed = allowed | (k_idx == q_idx)[None]                  # a padded row still attends to itself (no NaNs)
    return allowed.unsqueeze(1)

def run_local_conv(conv_module, x, tail, use_cache):
    """The depthwise causal conv of ParallelLocalGlobalBlock with an optional incremental state (last k-1 inputs)."""
    k = conv_module.kernel_size
    if tail is None:
        out = conv_module(x)
        if not use_cache:
            return out, None
        return out, F.pad(x, (0, 0, k - 1, 0))[:, -(k - 1):]
    window = torch.cat([tail, x], dim=1)
    hidden = conv_module.conv(window.transpose(1, 2)).transpose(1, 2)
    return conv_module.drop(conv_module.proj(F.gelu(hidden))), window[:, -(k - 1):]

def run_block(block, x, position_ids, attn_mask, cache, use_cache, token_mask):
    """Plain blocks are called as-is. ParallelLocalGlobalBlock gets a hand-written forward that carries the conv state
    (its own forward returns no cache); the weights are the Phase 1 weights."""
    if not isinstance(block, ParallelLocalGlobalBlock):
        return block(x, position_ids, attn_mask, cache, use_cache)
    attn_cache, conv_tail = (None, None) if cache is None else cache
    normed = block.ln1(x)
    attn_result = block.attn(normed, position_ids=position_ids, attn_mask=attn_mask, kv_cache=attn_cache, use_cache=use_cache)
    attn_out, new_attn_cache = attn_result if use_cache else (attn_result, None)
    conv_in = normed if token_mask is None else normed * token_mask.unsqueeze(-1).to(normed.dtype)   # padding = zeros
    conv_out, new_tail = run_local_conv(block.conv, conv_in, conv_tail, use_cache)
    x = x + block.alpha * attn_out + block.beta * conv_out
    x = x + block.ffn(block.ln2(x))
    return x, ((new_attn_cache, new_tail) if use_cache else None)

def decoder_hidden(model, x, position_ids, attn_mask, caches, use_cache, token_mask, use_ckpt):
    """Embeddings [B, T, D] -> final-norm hidden states [B, T, D] (+ per-layer caches)."""
    backbone = model.backbone
    if backbone.use_sin:
        x = x + backbone.pos_emb[position_ids]
    new_caches = []
    for i, block in enumerate(backbone.blocks):
        if use_ckpt and torch.is_grad_enabled():
            x = checkpoint(lambda h, blk=block: run_block(blk, h, position_ids, attn_mask, None, False, token_mask)[0], x, use_reentrant=False)
            new_caches.append(None)
        else:
            x, cache = run_block(block, x, position_ids, attn_mask, None if caches is None else caches[i], use_cache, token_mask)
            new_caches.append(cache)
    return backbone.ln_f(x), new_caches

def train_hidden(model, ids, use_ckpt):
    """Right-padded batch [B, L] -> hidden states [B, L, D] for the real tokens (prefix positions removed)."""
    x = embed_tokens(model, ids, with_prefix=True)
    B, total = x.size(0), x.size(1)
    position_ids = torch.arange(total, device=ids.device).unsqueeze(0).expand(B, total)
    hidden, _ = decoder_hidden(model, x, position_ids, None, None, False, None, use_ckpt)
    return hidden[:, model.n_prefix:]

# ── model factory ──
def freeze(module):
    for p in module.parameters():
        p.requires_grad = False

def build_model(base_lm, exp):
    backbone = copy.deepcopy(base_lm)
    if exp.block_stride > 1:                                         # KD student: keep every n-th block
        keep = list(range(0, len(backbone.blocks), exp.block_stride))
        backbone.blocks = nn.ModuleList([backbone.blocks[i] for i in keep])
    d_model = backbone.cfg.d_model
    soft_prefix = None

    if exp.method == "full":
        pass                                                         # everything trainable
    elif exp.method == "partial":                                    # top-n blocks + final norm
        freeze(backbone)
        for block in backbone.blocks[-exp.n_top:]:
            for p in block.parameters(): p.requires_grad = True
        for p in backbone.ln_f.parameters(): p.requires_grad = True
    elif exp.method in PEFT_WRAPPERS:
        freeze(backbone)
        wrap_attention_projections(backbone, PEFT_WRAPPERS[exp.method], exp.rank, exp.alpha)
    elif exp.method in ("houlsby", "pfeiffer"):
        freeze(backbone)
        add_adapters(backbone, d_model, exp.adapter_dim, exp.method)
    elif exp.method in ("prompt", "prefix"):
        assert exp.n_prefix <= RESERVED_PREFIX
        freeze(backbone)
        soft_prefix = SoftPrefix(exp.n_prefix, d_model, backbone.tok_emb.weight.detach(), reparam=(exp.method == "prefix"))
    else:
        raise ValueError(f"unknown method: {exp.method}")
    return SummarizerLM(backbone, soft_prefix).to(DEVICE)

def count_params(model):
    """(total, trainable) in millions. Quantised int8 weights (QLoRA) are counted as parameters."""
    total = sum(p.numel() for p in model.parameters()) + sum(b.numel() for n, b in model.named_buffers() if n.endswith("w_int8"))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total / 1e6, trainable / 1e6

def model_size_mb(model):
    """Real storage of the weights (int8 weights of QLoRA count 1 byte)."""
    n_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    n_bytes += sum(b.numel() * b.element_size() for n, b in model.named_buffers() if n.endswith(("w_int8", "w_scale")))
    return n_bytes / 2**20

## 7. Generation

In [11]:
BANNED_TOKENS = [PAD, UNK, BOS]

def banned_by_ngram(generated, n):
    """Next tokens that would repeat an n-gram already present in `generated`."""
    if len(generated) < n:
        return []
    context = tuple(generated[-(n - 1):])
    return [generated[i + n - 1] for i in range(len(generated) - n + 1) if tuple(generated[i: i + n - 1]) == context]

@torch.no_grad()
def generate_batch(model, batch_prompts, max_new, min_new, no_repeat, rep_penalty):
    """Greedy decoding for left-padded prompts. Stops per sequence at </s>. Returns token ids WITHOUT </s>."""
    backbone, n_prefix = model.backbone, model.n_prefix
    B, L = len(batch_prompts), max(len(p) for p in batch_prompts)
    assert n_prefix + L + max_new <= backbone.cfg.max_seq_len, "prompt + generation exceeds the Phase 1 context window"

    ids = torch.full((B, L), PAD, dtype=torch.long)
    token_mask = torch.zeros(B, L, dtype=torch.bool)
    for b, prompt in enumerate(batch_prompts):
        ids[b, L - len(prompt):] = torch.tensor(prompt)
        token_mask[b, L - len(prompt):] = True
    ids, token_mask = ids.to(DEVICE), token_mask.to(DEVICE)
    full_mask = torch.cat([torch.ones(B, n_prefix, dtype=torch.bool, device=DEVICE), token_mask], dim=1)

    position_ids = (full_mask.long().cumsum(-1) - 1).clamp(min=0)
    next_position = position_ids[:, -1] + 1
    with autocast("cuda", dtype=torch.float16):                       # prefill
        hidden, caches = decoder_hidden(model, embed_tokens(model, ids, True), position_ids,
                                        build_attn_mask(full_mask, 0, full_mask.size(1)), None, True, full_mask, False)
        logits = backbone.head(hidden[:, -1])

    generated = [[] for _ in range(B)]
    finished = [False] * B
    for step in range(max_new):
        scores = logits.float()
        scores[:, BANNED_TOKENS] = float("-inf")
        if step < min_new:
            scores[:, EOS] = float("-inf")
        for b in range(B):
            if rep_penalty != 1.0 and generated[b]:                   # penalise tokens already generated (not the prompt)
                seen = torch.tensor(sorted(set(generated[b])), device=DEVICE)
                values = scores[b, seen]
                scores[b, seen] = torch.where(values < 0, values * rep_penalty, values / rep_penalty)
            if no_repeat > 0:
                banned = banned_by_ngram(generated[b], no_repeat)
                if banned:
                    scores[b, banned] = float("-inf")
        chosen = scores.argmax(dim=-1).tolist()
        for b, token in enumerate(chosen):
            if finished[b]:
                continue
            if token == EOS:
                finished[b] = True
            else:
                generated[b].append(token)
        if all(finished) or step == max_new - 1:
            break

        feed = torch.tensor([[EOS if finished[b] else chosen[b]] for b in range(B)], device=DEVICE)
        full_mask = torch.cat([full_mask, torch.ones(B, 1, dtype=torch.bool, device=DEVICE)], dim=1)
        with autocast("cuda", dtype=torch.float16):
            hidden, caches = decoder_hidden(model, embed_tokens(model, feed, False), next_position[:, None],
                                            build_attn_mask(full_mask, full_mask.size(1) - 1, 1), caches, True,
                                            torch.ones(B, 1, dtype=torch.bool, device=DEVICE), False)
            logits = backbone.head(hidden[:, -1])
        next_position = next_position + 1
    return generated

def generate_ids(model, prompts, max_new_tokens, min_new_tokens, no_repeat_ngram, repetition_penalty, desc):
    """Batched generation; prompts are processed longest-first and results come back in the original order."""
    model.eval()
    order = sorted(range(len(prompts)), key=lambda i: -len(prompts[i]))
    outputs = [None] * len(prompts)
    for start in tqdm(range(0, len(order), GEN_BATCH_DOCS), desc=desc, leave=False):
        rows = order[start: start + GEN_BATCH_DOCS]
        generated = generate_batch(model, [prompts[i] for i in rows], max_new_tokens, min_new_tokens, no_repeat_ngram, repetition_penalty)
        for i, tokens in zip(rows, generated):
            outputs[i] = tokens
    return outputs

def postprocess_summary(text, dedupe):
    """Single paragraph; optionally drop sentences that were generated twice (redundancy reduction)."""
    text = re.sub(r"\s+", " ", text).strip()
    if not dedupe:
        return text
    kept, seen = [], set()
    for sentence in re.split(r"(?<=[.!?])\s+", text):
        if sentence.lower() not in seen:
            seen.add(sentence.lower()); kept.append(sentence)
    return " ".join(kept)

def summarize_direct(model, split, doc_inputs, exp):
    prompts = [build_prompt(tokens) for tokens in doc_inputs]
    generated = generate_ids(model, prompts, exp.gen_max_new, exp.gen_min_new, exp.gen_no_repeat, exp.gen_rep_penalty, "summarise")
    summaries = [postprocess_summary(tokenizer.decode(tokens), exp.gen_dedupe) for tokens in generated]
    assert all(s for s in summaries), "empty generated summary"
    return summaries

def segment_starts(n_tokens):
    starts = list(range(0, n_tokens, HIER_SEGMENT_TOKENS))
    if len(starts) > HIER_MAX_SEGMENTS:
        starts = [starts[i] for i in np.linspace(0, len(starts) - 1, HIER_MAX_SEGMENTS).round().astype(int)]
    return starts

def summarize_hierarchical(model, split, exp):
    """Pass 1: summarise every segment of the whole case. Pass 2: summarise the concatenated segment summaries."""
    assert HIER_MAX_SEGMENTS * (HIER_SEGMENT_SUMMARY_TOKENS + len(NL_IDS)) <= DOC_BUDGET
    segment_prompts, owner = [], []
    for d, tokens in enumerate(split.doc_tokens):
        for start in segment_starts(len(tokens)):
            segment_prompts.append(build_prompt(tokens[start: start + HIER_SEGMENT_TOKENS])); owner.append(d)
    segment_outputs = generate_ids(model, segment_prompts, HIER_SEGMENT_SUMMARY_TOKENS, 16, exp.gen_no_repeat, exp.gen_rep_penalty, "segment summaries")
    combined = [[] for _ in split.doc_tokens]
    for d, tokens in zip(owner, segment_outputs):
        combined[d] += tokens + NL_IDS
    second_inputs = [np.array(c[:DOC_BUDGET], dtype=np.int32) for c in combined]
    return summarize_direct(model, split, second_inputs, exp)

def summarize(model, split, doc_inputs, exp, mode):
    if mode == "direct":
        return summarize_direct(model, split, doc_inputs, exp)
    assert mode == "hierarchical", mode
    return summarize_hierarchical(model, split, exp)

## 8. Training Loop

In [12]:
def checkpoint_path(name):
    return f"{OUT}/ckpt_{name}.pt"     # latest epoch (resume point), overwritten every epoch

def best_path(name):
    return f"{OUT}/best_{name}.pt"     # best epoch only, kept separately

def save_atomic(obj, path):
    """Write to a temp file, then rename: an interruption can never corrupt the previous file."""
    torch.save(obj, path + ".tmp")
    os.replace(path + ".tmp", path)

def get_rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all()}

def set_rng_state(state):
    random.setstate(state["python"]); np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"]); torch.cuda.set_rng_state_all(state["cuda"])

def save_best(exp, best_state, best_metric, epoch):
    save_atomic({"state": best_state, "metric": best_metric, "epoch": epoch}, best_path(exp.name))

def save_epoch_checkpoint(exp, model, snapshot_keys, optimizer, scaler, adalora_ctrl, epoch, global_step, history):
    """Everything needed to continue training from the end of `epoch` (epochs completed so far)."""
    model_state = {k: v.detach().cpu() for k, v in model.state_dict().items() if k in snapshot_keys}
    save_atomic({"config": vars(exp), "epoch": epoch, "global_step": global_step, "history": history,
                 "model": model_state, "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                 "adalora_importance": None if adalora_ctrl is None else [imp.cpu() for imp in adalora_ctrl.importance],
                 "rng": get_rng_state()}, checkpoint_path(exp.name))

def resume_if_checkpointed(exp, model, optimizer, scaler, adalora_ctrl, fresh_history, worst_metric):
    """Returns (start_epoch, global_step, history, best_metric, best_state).
    Fresh start values are returned when this experiment has no checkpoint yet."""
    if not os.path.exists(checkpoint_path(exp.name)):
        return 0, 0, fresh_history, worst_metric, None

    ckpt = torch.load(checkpoint_path(exp.name), map_location="cpu", weights_only=False)
    assert ckpt["config"] == vars(exp), f"{checkpoint_path(exp.name)} was written by a different configuration; delete it (and best_{exp.name}.pt) to start over"
    outcome = model.load_state_dict(ckpt["model"], strict=False)
    assert not outcome.unexpected_keys, outcome.unexpected_keys
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    if adalora_ctrl is not None:
        for importance, saved in zip(adalora_ctrl.importance, ckpt["adalora_importance"]):
            importance.copy_(saved)
    set_rng_state(ckpt["rng"])

    best = torch.load(best_path(exp.name), map_location="cpu", weights_only=False)
    print(f"[{exp.name}] resumed after epoch {ckpt['epoch']}/{exp.epochs} (best so far {best['metric']:.4f} at epoch {best['epoch']})")
    return ckpt["epoch"], ckpt["global_step"], ckpt["history"], best["metric"], best["state"]

In [13]:
def ngram_set(words, n):
    return {tuple(words[i: i + n]) for i in range(len(words) - n + 1)}

def repeated_ngram_rate(summary, n=3):
    """Share of the summary's n-grams that are repeats of an earlier n-gram in the same summary (0 = no repetition)."""
    words = summary.lower().split()
    grams = [tuple(words[i: i + n]) for i in range(len(words) - n + 1)]
    return 0.0 if not grams else 1.0 - len(set(grams)) / len(grams)

def novel_ngram_rate(summary, document, n=3):
    """Share of the summary's n-grams that do NOT occur in the document (abstractiveness)."""
    grams = ngram_set(summary.lower().split(), n)
    return 0.0 if not grams else len(grams - ngram_set(document.lower().split(), n)) / len(grams)

def number_recall(summary, reference):
    """Recall of the numbers (years, section numbers, amounts) of the reference: a cheap legal-fact coverage proxy."""
    ref_numbers = set(re.findall(r"\d+", reference))
    return float("nan") if not ref_numbers else len(ref_numbers & set(re.findall(r"\d+", summary))) / len(ref_numbers)

def compute_summary_metrics(predictions, references, documents):
    rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    rouge_scores = [rouge.score(ref, pred) for pred, ref in zip(predictions, references)]

    # BERTScore (primary metric of the competition). The scorer's own pretrained encoder is used for EVALUATION only.
    scorer = BERTScorer(lang="en", device=DEVICE)
    precision, recall, f1 = scorer.score(predictions, references, batch_size=16)
    del scorer; gc.collect(); torch.cuda.empty_cache()

    return {"bert_F1": f1.mean().item(), "bert_P": precision.mean().item(), "bert_R": recall.mean().item(),
            "rouge1": float(np.mean([s["rouge1"].fmeasure for s in rouge_scores])),
            "rougeL": float(np.mean([s["rougeL"].fmeasure for s in rouge_scores])),
            "avg_words": float(np.mean([len(p.split()) for p in predictions])),
            "ref_avg_words": float(np.mean([len(r.split()) for r in references])),
            "repeat_3gram": float(np.mean([repeated_ngram_rate(p) for p in predictions])),
            "novel_3gram": float(np.mean([novel_ngram_rate(p, d) for p, d in zip(predictions, documents)])),
            "number_recall": float(np.nanmean([number_recall(p, r) for p, r in zip(predictions, references)]))}

In [14]:
def selected_logits(model, hidden, labels):
    """Next-token logits only at the positions whose label is a summary token: [N, V] and targets [N]."""
    select = labels[:, 1:] != -100
    return model.backbone.head(hidden[:, :-1][select]), labels[:, 1:][select]

def summary_loss(model, batch, exp, teacher):
    """Cross-entropy on summary tokens (+ KL to the teacher's token distribution for knowledge distillation)."""
    ids, labels = batch["ids"].to(DEVICE), batch["labels"].to(DEVICE)
    with autocast("cuda", dtype=torch.float16):
        logits, targets = selected_logits(model, train_hidden(model, ids, GRAD_CKPT), labels)
    logits = logits.float()
    ce = F.cross_entropy(logits, targets)
    if teacher is None:
        return ce
    with torch.no_grad(), autocast("cuda", dtype=torch.float16):
        teacher_logits, _ = selected_logits(teacher, train_hidden(teacher, ids, False), labels)
    T = exp.kd_temperature
    kd = F.kl_div(F.log_softmax(logits / T, -1), F.softmax(teacher_logits.float() / T, -1), reduction="batchmean") * T * T
    return exp.kd_hard_weight * ce + (1.0 - exp.kd_hard_weight) * kd

@torch.no_grad()
def evaluate_nll(model, dataset):
    """Teacher-forced negative log-likelihood per summary token."""
    model.eval()
    loader = DataLoader(dataset, batch_size=EVAL_BATCH_DOCS, shuffle=False, collate_fn=collate_lm)
    total, count = 0.0, 0
    for batch in tqdm(loader, desc="val nll", leave=False):
        ids, labels = batch["ids"].to(DEVICE), batch["labels"].to(DEVICE)
        with autocast("cuda", dtype=torch.float16):
            logits, targets = selected_logits(model, train_hidden(model, ids, False), labels)
        total += F.cross_entropy(logits.float(), targets, reduction="sum").item()
        count += targets.numel()
    return total / count

def snapshot_names(model):
    """Parameters (and AdaLoRA masks) that must be restored to recover a given epoch."""
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}
    return trainable | {k for k in model.state_dict() if k.endswith(".mask")}

def train_model(model, exp, train_ds, val_ds, teacher):
    loader = DataLoader(train_ds, batch_size=exp.batch_docs, shuffle=True, collate_fn=collate_lm, pin_memory=True)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=exp.lr, weight_decay=WEIGHT_DECAY)

    steps_per_epoch = math.ceil(len(loader) / exp.grad_accum)
    total_steps = exp.epochs * steps_per_epoch
    warmup_steps = int(WARMUP_FRAC * total_steps)
    scaler = GradScaler("cuda")

    adalora_modules = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]
    adalora_ctrl = AdaLoRAController(adalora_modules, total_steps, exp.adalora_final_ratio) if adalora_modules else None

    fresh_history = {"val_nll": [], "loss": [], "train_sec": 0.0}
    start_epoch, global_step, history, best_nll, best_state = resume_if_checkpointed(
        exp, model, optimizer, scaler, adalora_ctrl, fresh_history, worst_metric=float("inf"))

    for epoch in range(start_epoch, exp.epochs):
        t0 = time.time()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_loss = 0.0

        for step_in_epoch, batch in enumerate(tqdm(loader, desc=f"{exp.name} ep {epoch + 1}/{exp.epochs}", leave=False)):
            loss = summary_loss(model, batch, exp, teacher)
            if adalora_modules:                                      # keep P, Q near-orthogonal
                loss = loss + ADALORA_ORTH_COEF * torch.stack([m.orth_loss() for m in adalora_modules]).mean()
            scaler.scale(loss / exp.grad_accum).backward()
            epoch_loss += loss.item()

            is_boundary = (step_in_epoch + 1) % exp.grad_accum == 0 or step_in_epoch == len(loader) - 1
            if is_boundary:
                # linear warm-up, then cosine decay
                if global_step < warmup_steps:
                    lr_frac = (global_step + 1) / max(1, warmup_steps)
                else:
                    progress = (global_step - warmup_steps) / max(1, total_steps - warmup_steps)
                    lr_frac = 0.5 * (1.0 + math.cos(math.pi * progress))
                for group in optimizer.param_groups:
                    group["lr"] = exp.lr * lr_frac

                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                if adalora_ctrl is not None and torch.isfinite(grad_norm):
                    adalora_ctrl.update(global_step)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

        val_nll = evaluate_nll(model, val_ds)                        # model selection: lowest validation NLL
        history["val_nll"].append(val_nll); history["loss"].append(epoch_loss / len(loader)); history["train_sec"] += time.time() - t0
        extra = f" | active AdaLoRA rank {adalora_ctrl.active_rank()}/{adalora_ctrl.initial_budget}" if adalora_ctrl else ""
        print(f"[{exp.name}] ep {epoch + 1}/{exp.epochs} train loss {history['loss'][-1]:.4f} | val NLL {val_nll:.4f} (ppl {math.exp(val_nll):.2f}){extra}")

        # ── checkpoints: best model (only when improved) first, then the resume point ──
        snapshot_keys = snapshot_names(model)
        if val_nll < best_nll:
            best_nll = val_nll
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in snapshot_keys}
            save_best(exp, best_state, best_nll, epoch + 1)
        save_epoch_checkpoint(exp, model, snapshot_keys, optimizer, scaler, adalora_ctrl, epoch + 1, global_step, history)

    model.load_state_dict(best_state, strict=False)                  # restore best epoch
    history["best_val_nll"] = best_nll
    return history

## 9. Experiments

In [15]:
@dataclass
class Experiment:
    name: str
    base_model: str               # one of PHASE1_CONFIGS
    init_from: str                # phase1 | dapt (Phase 1 weights after domain-adaptive pretraining)
    method: str                   # full | partial | lora | lorma | qlora | adalora | houlsby | pfeiffer | prompt | prefix
    strategy: str                 # head | tail | head_tail | uniform | sentence_select
    epochs: int
    lr: float
    batch_docs: int
    grad_accum: int
    rank: int                     # LoRA / LoRMA / QLoRA / AdaLoRA rank
    alpha: float
    n_top: int                    # blocks trained by "partial"
    adapter_dim: int
    n_prefix: int                 # soft tokens for prompt / prefix tuning
    adalora_final_ratio: float    # fraction of AdaLoRA ranks kept after pruning
    block_stride: int             # 1 = all blocks; 2 = every 2nd block (KD student)
    teacher_name: str             # experiment whose trained model is distilled (None = no KD)
    kd_temperature: float
    kd_hard_weight: float         # weight of the hard-label loss in the KD objective
    gen_max_new: int              # decoding: maximum summary tokens (<= MAX_SUMMARY_TOKENS)
    gen_min_new: int              # decoding: minimum summary tokens
    gen_no_repeat: int            # decoding: block repeated n-grams of this size (0 = off)
    gen_rep_penalty: float        # decoding: repetition penalty on generated tokens (1.0 = off)
    gen_dedupe: bool              # post-process: drop duplicated sentences
    write_test: bool              # also generate the test summaries and write the submission

BASE = Experiment(name="", base_model=MAIN_MODEL, init_from="phase1", method="lorma", strategy="head_tail", epochs=4, lr=2e-4,
                  batch_docs=BATCH_DOCS, grad_accum=GRAD_ACCUM, rank=16, alpha=32.0, n_top=4, adapter_dim=64, n_prefix=16,
                  adalora_final_ratio=0.5, block_stride=1, teacher_name=None, kd_temperature=2.0, kd_hard_weight=0.5,
                  gen_max_new=MAX_SUMMARY_TOKENS, gen_min_new=32, gen_no_repeat=6, gen_rep_penalty=1.0, gen_dedupe=True,
                  write_test=True)

def make_exp(name, **overrides):
    return replace(BASE, name=name, **overrides)

def result_path(name):  return f"{OUT}/result_{name}.json"
def weights_path(name): return f"{OUT}/weights_{name}.pt"

def load_trained_model(name):
    """Rebuild a finished experiment from its saved config + trained weights."""
    exp = Experiment(**json.load(open(result_path(name)))["config"])
    model = build_model(load_base(exp.base_model, exp.init_from), exp)
    outcome = model.load_state_dict(torch.load(weights_path(name), map_location="cpu", weights_only=False), strict=False)
    assert not outcome.unexpected_keys, outcome.unexpected_keys
    return model, exp

def write_submission(split, summaries, name):
    submission = [{"id": case_id, "summary": summary} for case_id, summary in zip(split.ids, summaries)]
    assert len(submission) == len(split.ids) and all(s["summary"].strip() for s in submission)
    json_path, zip_path = f"{OUT}/submission_{name}.json", f"{OUT}/submission_{name}.zip"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(json_path, "prediction.json")
    print(f"  -> {zip_path}  (avg {np.mean([len(s['summary'].split()) for s in submission]):.0f} words/summary)")

def write_test_submission(name, mode="direct"):
    """Generate the test summaries of a finished experiment (use for experiments that ran with write_test=False)."""
    model, exp = load_trained_model(name)
    summaries = summarize(model, test_split, get_doc_inputs("test", test_split, exp.strategy), exp, mode)
    write_submission(test_split, summaries, name if mode == "direct" else f"{name}_{mode}")
    model.cpu(); del model; gc.collect(); torch.cuda.empty_cache()

def evaluate_variant(name, mode, **decoding_overrides):
    """Re-score a finished experiment on the validation subset with other decoding settings / hierarchical mode."""
    model, exp = load_trained_model(name)
    exp = replace(exp, **decoding_overrides)
    n_gen = min(N_VAL_GEN, len(val_split.ids))
    subset = select_rows(val_split, range(n_gen))
    doc_inputs = get_doc_inputs("val", val_split, exp.strategy)[:n_gen]
    torch.cuda.synchronize(); t0 = time.perf_counter()
    summaries = summarize(model, subset, doc_inputs, exp, mode)
    torch.cuda.synchronize(); sec_per_doc = (time.perf_counter() - t0) / n_gen
    model.cpu(); del model; gc.collect(); torch.cuda.empty_cache()
    return {**compute_summary_metrics(summaries, subset.summaries, subset.texts), "sec_per_doc": sec_per_doc}

def run_experiment(exp):
    """Build -> train -> generate + score on validation -> (optionally) write the test submission -> save results."""
    if os.path.exists(result_path(exp.name)):
        print(f"[{exp.name}] finished earlier -> skipped"); return

    set_seed(SEED)
    model = build_model(load_base(exp.base_model, exp.init_from), exp)
    teacher = None
    if exp.teacher_name is not None:
        teacher, _ = load_trained_model(exp.teacher_name)
        teacher.eval(); freeze(teacher)

    train_ds = SummaryDataset(train_split, get_doc_inputs("train", train_split, exp.strategy))
    val_ds   = SummaryDataset(val_split,   get_doc_inputs("val",   val_split,   exp.strategy))

    total_m, trainable_m = count_params(model)
    print(f"\n=== {exp.name}: total {total_m:.1f}M | trainable {trainable_m:.2f}M ({100 * trainable_m / total_m:.1f}%)")
    torch.cuda.reset_peak_memory_stats()
    history = train_model(model, exp, train_ds, val_ds, teacher)
    names = snapshot_names(model)
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items() if k in names}, weights_path(exp.name))
    if teacher is not None:
        teacher.cpu(); del teacher; gc.collect(); torch.cuda.empty_cache()

    # generation on a validation subset (timed) + scores
    n_gen = min(N_VAL_GEN, len(val_split.ids))
    val_subset = select_rows(val_split, range(n_gen))
    torch.cuda.synchronize(); t0 = time.perf_counter()
    val_summaries = summarize_direct(model, val_subset, get_doc_inputs("val", val_split, exp.strategy)[:n_gen], exp)
    torch.cuda.synchronize(); infer_sec_per_doc = (time.perf_counter() - t0) / n_gen
    peak_gb = torch.cuda.max_memory_allocated() / 2**30          # training + generation peak (measured before the BERTScore encoder is loaded)
    metrics = compute_summary_metrics(val_summaries, val_subset.summaries, val_subset.texts)
    with open(f"{OUT}/val_predictions_{exp.name}.json", "w", encoding="utf-8") as f:
        json.dump([{"id": i, "summary": p, "reference": r} for i, p, r in zip(val_subset.ids, val_summaries, val_subset.summaries)], f, ensure_ascii=False, indent=2)

    if exp.write_test:
        write_submission(test_split, summarize_direct(model, test_split, get_doc_inputs("test", test_split, exp.strategy), exp), exp.name)

    result = {"name": exp.name, "config": asdict(exp), "total_M": total_m, "trainable_M": trainable_m, "size_MB": model_size_mb(model),
              "metrics": metrics, "val_nll": history["best_val_nll"], "train_min": history["train_sec"] / 60,
              "infer_s_per_doc": infer_sec_per_doc, "peak_GB": peak_gb, "history": history}
    with open(result_path(exp.name), "w") as f:
        json.dump(result, f, indent=2)
    print(f"### {exp.name}: trainable {trainable_m:.2f}M/{total_m:.1f}M | val NLL {history['best_val_nll']:.4f} | BERTScore F1 {metrics['bert_F1']:.4f} "
          f"| ROUGE-1 {metrics['rouge1']:.4f} ROUGE-L {metrics['rougeL']:.4f} | {metrics['avg_words']:.0f} words (ref {metrics['ref_avg_words']:.0f})")

    model.cpu(); del model; gc.collect(); torch.cuda.empty_cache()

In [16]:
class WindowDataset(Dataset):
    """Plain language-modelling windows: <s> + up to TRAIN_BLOCK_SIZE-1 document tokens; every token is a target."""
    def __init__(self, windows):
        self.windows = windows
    def __len__(self):
        return len(self.windows)
    def __getitem__(self, i):
        ids = [BOS] + self.windows[i].tolist()
        return {"ids": ids, "labels": ids}

def make_windows(token_arrays, min_len=64):
    size = TRAIN_BLOCK_SIZE - 1
    return [t[s: s + size] for t in token_arrays for s in range(0, len(t), size) if len(t[s: s + size]) >= min_len]

def domain_adapt(model_name, epochs, lr, batch_docs, grad_accum):
    """Full-model continued pretraining on TRAIN documents only (val documents are held out, test documents are never used).
    Saves the adapted Phase 1 weights; later experiments start from them with init_from='dapt'."""
    if os.path.exists(dapt_path(model_name)):
        print(f"[dapt_{model_name}] finished earlier -> skipped"); return
    set_seed(SEED)
    model = SummarizerLM(load_phase1_model(model_name), None).to(DEVICE)
    train_ds = WindowDataset(make_windows(train_split.doc_tokens))
    val_ds   = WindowDataset(make_windows(val_split.doc_tokens)[:100])
    print(f"[dapt_{model_name}] {len(train_ds)} train windows | LM loss on held-out documents BEFORE: {evaluate_nll(model, val_ds):.4f}")
    exp = make_exp(f"dapt_{model_name}", base_model=model_name, method="full", epochs=epochs, lr=lr, batch_docs=batch_docs, grad_accum=grad_accum)
    history = train_model(model, exp, train_ds, val_ds, None)
    print(f"[dapt_{model_name}] LM loss on held-out documents AFTER: {history['best_val_nll']:.4f}")
    torch.save(model.backbone.state_dict(), dapt_path(model_name))
    model.cpu(); del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
lm = SummarizerLM(load_phase1_model(MAIN_MODEL), None).to(DEVICE).eval()
x = torch.tensor([[BOS] + train_split.doc_tokens[0][:512].tolist()], device=DEVICE)
with torch.no_grad(), autocast("cuda", dtype=torch.float16):
    logits = lm.backbone.head(train_hidden(lm, x, False))
lm_loss = F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.size(-1)), x[:, 1:].reshape(-1)).item()
print(f"Phase-1 LM loss on a train document: {lm_loss:.3f} (random init ~ {math.log(VOCAB_SIZE):.1f})")
print("example prompt tail:", repr(tokenizer.decode(build_prompt(train_split.doc_tokens[0][:60])[-30:])))
del lm; gc.collect(); torch.cuda.empty_cache()

## 10. Smoke Test

In [ ]:
if SMOKE_TEST:
    quick = dict(epochs=1, batch_docs=1, grad_accum=2, gen_max_new=48, gen_min_new=8)
    run_experiment(make_exp("smoke_full", method="full", **quick))
    run_experiment(make_exp("smoke_kd", method="full", block_stride=2, teacher_name="smoke_full", **quick))
    for method in ["partial", "lora", "lorma", "qlora", "adalora", "houlsby", "pfeiffer", "prompt", "prefix"]:
        run_experiment(make_exp(f"smoke_{method}", method=method, **quick))
    for model_name in PHASE1_CONFIGS:                                 # exercises cached generation of every architecture (incl. conv)
        run_experiment(make_exp(f"smoke_lorma_{model_name}", base_model=model_name, **quick))
    for strategy in ["head", "tail", "uniform", "sentence_select"]:
        run_experiment(make_exp(f"smoke_strategy_{strategy}", strategy=strategy, write_test=False, **quick))
    domain_adapt(MAIN_MODEL, epochs=1, lr=3e-5, batch_docs=1, grad_accum=2)
    run_experiment(make_exp("smoke_dapt_lorma", init_from="dapt", write_test=False, **quick))
    print(evaluate_variant("smoke_lorma", "hierarchical", gen_max_new=48, gen_min_new=8))
    print(evaluate_variant("smoke_lorma", "direct", gen_max_new=48, gen_min_new=8, gen_rep_penalty=1.15, gen_dedupe=False))
    write_test_submission("smoke_lorma")

## 11. Runs

In [19]:
run_experiment(make_exp("full_finetune", method="full", epochs=3, lr=3e-5))


=== full_finetune: total 190.0M | trainable 190.02M (100.0%)
[full_finetune] resumed after epoch 3/3 (best so far 2.0421 at epoch 2)


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  -> ./cs781_phase6/submission_full_finetune.zip  (avg 219 words/summary)
### full_finetune: trainable 190.02M/190.0M | val NLL 2.0421 | BERTScore F1 0.8379 | ROUGE-1 0.3441 ROUGE-L 0.2043 | 217 words (ref 790)


In [21]:
# Full fine-tuning: quality ceiling AND the KD teacher
# run_experiment(make_exp("full_finetune", method="full", epochs=1, lr=3e-5))

run_experiment(make_exp("lora_r16",  base_model= "mla-sin",   method="lora", epochs=1,     lr=2e-4))
# run_experiment(make_exp("qlora_r16",    method="qlora",    lr=2e-4))
run_experiment(make_exp("adalora_r16", base_model= "mla-sin",  method="adalora", epochs=1,  lr=2e-4, adalora_final_ratio=0.5))



=== lora_r16: total 92.7M | trainable 1.14M (1.2%)


[lora_r16] ep 1/1 train loss 4.0188 | val NLL 4.0023 (ppl 54.72)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  -> ./cs781_phase6/submission_lora_r16.zip  (avg 228 words/summary)
### lora_r16: trainable 1.14M/92.7M | val NLL 4.0023 | BERTScore F1 0.7932 | ROUGE-1 0.2574 ROUGE-L 0.1750 | 227 words (ref 790)

=== adalora_r16: total 92.7M | trainable 1.14M (1.2%)


[adalora_r16] ep 1/1 train loss 4.1210 | val NLL 4.1183 (ppl 61.46) | active AdaLoRA rank 480/960


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  -> ./cs781_phase6/submission_adalora_r16.zip  (avg 202 words/summary)
### adalora_r16: trainable 1.14M/92.7M | val NLL 4.1183 | BERTScore F1 0.7885 | ROUGE-1 0.2417 ROUGE-L 0.1697 | 214 words (ref 790)


In [22]:
run_experiment(make_exp("houlsby_64", base_model=  "swiglu-moe-rope-conv", epochs=1,  method="houlsby",  lr=5e-4, adapter_dim=64))
run_experiment(make_exp("pfeiffer_64", base_model= "swiglu-moe-rope-conv", epochs=1, method="pfeiffer", lr=5e-4, adapter_dim=64))


=== houlsby_64: total 199.7M | trainable 2.38M (1.2%)


[houlsby_64] ep 1/1 train loss 3.2490 | val NLL 3.2442 (ppl 25.64)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  -> ./cs781_phase6/submission_houlsby_64.zip  (avg 223 words/summary)
### houlsby_64: trainable 2.38M/199.7M | val NLL 3.2442 | BERTScore F1 0.8122 | ROUGE-1 0.2756 ROUGE-L 0.1788 | 223 words (ref 790)

=== pfeiffer_64: total 198.5M | trainable 1.19M (0.6%)


[pfeiffer_64] ep 1/1 train loss 3.2645 | val NLL 3.2652 (ppl 26.19)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  -> ./cs781_phase6/submission_pfeiffer_64.zip  (avg 225 words/summary)
### pfeiffer_64: trainable 1.19M/198.5M | val NLL 3.2652 | BERTScore F1 0.8102 | ROUGE-1 0.2697 ROUGE-L 0.1788 | 224 words (ref 790)


In [21]:
run_experiment(make_exp("prompt_16",  base_model= "gqa-rope",   method="prompt", epochs=1,   lr=3e-3, n_prefix=16))
run_experiment(make_exp("prefix_16", base_model=  "gqa-rope",  method="prefix", epochs=1,   lr=1e-3, n_prefix=16))


=== prompt_16: total 65.8M | trainable 0.01M (0.0%)


[prompt_16] ep 1/1 train loss 3.5333 | val NLL 3.5551 (ppl 34.99)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  -> ./cs781_phase6/submission_prompt_16.zip  (avg 219 words/summary)
### prompt_16: trainable 0.01M/65.8M | val NLL 3.5551 | BERTScore F1 0.7969 | ROUGE-1 0.2456 ROUGE-L 0.1670 | 224 words (ref 790)

=== prefix_16: total 66.9M | trainable 1.19M (1.8%)


[prefix_16] ep 1/1 train loss 3.5484 | val NLL 3.5728 (ppl 35.61)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  -> ./cs781_phase6/submission_prefix_16.zip  (avg 210 words/summary)
### prefix_16: trainable 1.19M/66.9M | val NLL 3.5728 | BERTScore F1 0.8003 | ROUGE-1 0.2180 ROUGE-L 0.1506 | 195 words (ref 790)


In [20]:
# Teacher: full_finetune (12 blocks). Student: same Phase 1 model keeping every 2nd block (6 blocks), distilled from the teacher's token distributions.
run_experiment(make_exp("kd_student_half_depth", method="full", block_stride=2, teacher_name="full_finetune",
                        epochs=1, lr=5e-5, kd_temperature=2.0, kd_hard_weight=0.5))


=== kd_student_half_depth: total 102.7M | trainable 102.69M (100.0%)


[kd_student_half_depth] ep 1/1 train loss 2.0238 | val NLL 2.6550 (ppl 14.23)


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  -> ./cs781_phase6/submission_kd_student_half_depth.zip  (avg 213 words/summary)
### kd_student_half_depth: trainable 102.69M/102.7M | val NLL 2.6550 | BERTScore F1 0.8267 | ROUGE-1 0.3167 ROUGE-L 0.1919 | 215 words (ref 790)
